# Horus · entrenar la cabeza de objetos (Colab / Kaggle)

**Esto NO reentrena el backbone.** El backbone no se entrena nunca: es un
ResNet-50 de ImageNet más un FPN que se sortea al azar y se congela ahí. Lo que
se perdió no es un modelo, es **un número de la lotería**.

Lo que hace este notebook es sortear un FPN nuevo, entrenar una cabeza contra
él, y **guardar los dos juntos, con huella** — para que no se pueda volver a
perder en silencio.

Al final te bajás **un solo archivo**, `head_best_solo.pt`, que ya lleva el
backbone adentro y no depende de nada al lado.

| | |
|---|---|
| lo que entrena | solo la cabeza (el backbone corre en `no_grad`) |
| referencia a batir | v2: mAP@0.50 = **0.5335**, época 41 |
| GPU | T4 alcanza. Kaggle da 30 h/semana y corta menos que Colab gratis |

**Hay dos compuertas duras.** El notebook se detiene solo si no se cumplen:
las correcciones tienen que estar en la rama que clonás, y el dataset tiene
que tener cajas de las 7 clases.


## 1 · GPU y entorno

In [ ]:
import subprocess, sys, os, json, pathlib
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "SIN GPU")
EN_COLAB = "google.colab" in sys.modules
EN_KAGGLE = os.path.exists("/kaggle")
print("entorno:", "Colab" if EN_COLAB else ("Kaggle" if EN_KAGGLE else "otro"))
import torch; print("torch", torch.__version__, "· cuda", torch.cuda.is_available())

## 2 · Clonar el repo — y verificar que traiga las correcciones

**Compuerta 1.** Si clonás una rama sin las correcciones del 15/09, esto se
detiene. Los tres arreglos que tienen que estar:

- `_ANCHOR_SIZES = ((16,20,25),(32,40,51),(64,81,102))` — los del v2. Con
  `((32,),(64,),(128,))` el v3 saldría peor justo en objetos chicos, que es
  donde vive una pistola a 16–47 px.
- `huella_backbone()` en `shared_backbone.py`.
- `--incrustar-backbone` en `exportar_objetos.py`.

Si falla: commiteá y pusheá esos cambios desde tu máquina primero.

In [ ]:
REPO = "https://github.com/Teo50000/Horus-AI.git"
RAMA = "Models"          # cambiala si pusheaste a otra

import shutil, subprocess, sys, os
if os.path.exists("Horus-AI"): shutil.rmtree("Horus-AI")
subprocess.run(["git","clone","--depth","1","-b",RAMA,REPO,"Horus-AI"], check=True)

RAIZ = os.path.abspath("Horus-AI/horus")
OBJ  = os.path.join(RAIZ, "04_cabezas", "objetos")
sys.path[:0] = [os.path.join(RAIZ,"03_backbone"), OBJ]

fallas = []
src = open(os.path.join(OBJ,"entrenar_objetos_cuda.py"), encoding="utf-8").read()
if "(16, 20, 25)" not in src:
    fallas.append("entrenar_objetos_cuda.py NO tiene los anchors del v2")
if "def huella_backbone" not in open(os.path.join(RAIZ,"03_backbone","shared_backbone.py"), encoding="utf-8").read():
    fallas.append("shared_backbone.py NO tiene huella_backbone()")
if "--incrustar-backbone" not in open(os.path.join(OBJ,"exportar_objetos.py"), encoding="utf-8").read():
    fallas.append("exportar_objetos.py NO tiene --incrustar-backbone")

if fallas:
    for f in fallas: print("  ✗", f)
    raise SystemExit("\nLa rama '%s' no trae las correcciones del 15/09.\n"
                     "Commiteá y pusheá desde tu máquina, o cambiá RAMA." % RAMA)
print("✓ la rama trae las tres correcciones")

## 3 · Checkpoints en Drive, para sobrevivir la desconexión

Colab gratis corta a las 12 h y por inactividad. El entrenador tiene
`--reanudar` de verdad —pesos, optimizador, scheduler y scaler—, así que con
los checkpoints en Drive una corrida cortada se retoma donde iba.

En Kaggle esto se saltea: el directorio de trabajo persiste entre sesiones.

In [ ]:
import os
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DEST = "/content/drive/MyDrive/horus_objetos_ckpt"
else:
    DEST = os.path.abspath("horus_objetos_ckpt")
os.makedirs(DEST, exist_ok=True)

ck_local = os.path.join(OBJ, "checkpoints")
if os.path.islink(ck_local) or os.path.exists(ck_local):
    if not os.path.islink(ck_local): shutil.rmtree(ck_local)
    else: os.unlink(ck_local)
os.symlink(DEST, ck_local)
print("checkpoints ->", DEST)
print("ya hay:", sorted(os.listdir(DEST)) or "(vacío, primera corrida)")

## 4 · Dependencias

In [ ]:
%pip install -q fiftyone kaggle pycocotools opencv-python-headless
print("listo")

## 5 · Credencial de Kaggle (solo para D-Fire)

D-Fire aporta humo y llama, que son dos de las siete clases. Subí tu
`kaggle.json` (Kaggle → Settings → API → Create New Token).

En Kaggle esto no hace falta.

In [ ]:
import os, json
if EN_COLAB:
    from google.colab import files
    print("Subí kaggle.json:")
    up = files.upload()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "wb") as f:
        f.write(list(up.values())[0])
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("ok" if os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")) else "sin credencial: D-Fire no se va a poder bajar")

## 6 · Bajar y armar el dataset

`mezcla_v1`: 7 clases, ~13.000 imágenes de train (incluye 4.000 negativos).
Tarda un rato largo — Open Images baja clase por clase.

In [ ]:
%cd {OBJ}
!python bajar_datasets.py --verificar
!python bajar_datasets.py --descargar d-fire pyro-sdis openimages
!python bajar_datasets.py --normalizar
!python bajar_datasets.py --armar

## 7 · Compuerta 2 · contar cajas por clase ANTES de gastar GPU

Hubo un bug en el que **todas** las etiquetas derivadas de Open Images
contenían únicamente la clase 2 (persona). Con ese dataset, `paquete`,
`celular`, `cuchillo` y `pistola` se entrenan **sin una sola caja positiva**:
el modelo aprende explícitamente que una caja de cartón no es un paquete.

La corrección ya está en `bajar_datasets.py`, pero esto lo verifica contando
en los `.txt`, no asumiéndolo. Si alguna clase quedó en cero, se detiene.

In [ ]:
import collections, pathlib
CLASES = ("humo","llama","persona","pistola","cuchillo","celular","paquete")
raiz = pathlib.Path(OBJ) / "datasets" / "mezcla_v1"
cuenta, imgs = collections.Counter(), 0
for split in ("train","val"):
    d = raiz / split / "labels"
    if not d.exists(): d = raiz / "labels" / split
    for t in d.glob("*.txt"):
        imgs += 1
        for linea in t.read_text().split("\n"):
            if linea.strip(): cuenta[int(linea.split()[0])] += 1

print(f"{imgs} etiquetas\n")
vacias = []
for i, n in enumerate(CLASES):
    c = cuenta.get(i, 0)
    print(f"  {i} {n:<10} {c:>7} cajas" + ("   <-- VACÍA" if c == 0 else ""))
    if c == 0: vacias.append(n)
if vacias:
    raise SystemExit(f"\nClases sin una sola caja: {vacias}.\n"
                     "Entrenar así les enseña al modelo que NO existen. "
                     "Revisá bajar_datasets.py antes de seguir.")
print("\n✓ las 7 clases tienen cajas")
!python diagnostico_clases.py

## 8 · Entrenar

Solo la cabeza: el backbone va congelado y en `no_grad`.

**No uses `--cachear-features` acá**: aprovecha que el backbone está congelado,
pero cuesta ~3 MB por foto — 13.000 imágenes son ~39 GB y no entran.

Si la sesión se corta, volvé a correr esta celda: retoma sola desde Drive.

In [ ]:
import os
EPOCAS, BATCH, TAM = 60, 8, 384
reanudar = os.path.join(DEST, "head_last.pt")
flag = f"--reanudar {reanudar}" if os.path.exists(reanudar) else ""
print("retomando" if flag else "empezando de cero")

!python entrenar_objetos_cuda.py --dataset datasets/mezcla_v1 \
    --epocas {EPOCAS} --batch {BATCH} --tam {TAM} --precision bf16 {flag}

## 9 · Incrustar el backbone y verificar

Acá es donde se cierra el agujero. Los tensores irreconstruibles —FPN,
`embed_head`, GRU: 3,4 M de parámetros, 13,6 MB— se meten **adentro** del
checkpoint, y queda con huella.

A partir de este archivo, perder un `backbone.pt` suelto ya no puede volver a
costar un modelo entero.

In [ ]:
%cd {OBJ}
!python exportar_objetos.py --incrustar-backbone checkpoints/head_best.pt \
    --backbone checkpoints/backbone.pt --salida checkpoints/head_best_solo.pt

# Prueba de verdad: se carga SIN el backbone al lado.
import shutil, tempfile, torch, os, sys
tmp = tempfile.mkdtemp()
shutil.copy(os.path.join(OBJ,"checkpoints","head_best_solo.pt"), tmp)
from objects_engine import EngineConfig, ObjectsEngine
m = ObjectsEngine(EngineConfig(pesos=os.path.join(tmp,"head_best_solo.pt"), verboso=True))
ck = torch.load(os.path.join(OBJ,"checkpoints","head_best_solo.pt"), map_location="cpu", weights_only=False)
print("\nhuella :", ck["backbone_huella"])
print("época  :", ck.get("epoca"), "· mAP@0.50 :", (ck.get("metricas") or {}).get("mAP@0.50"))
print("v2 de referencia: 0.5335")

## 10 · Bajarlo

`head_best_solo.pt` va a `horus/04_cabezas/objetos/modelos/`. El motor lo carga
solo, verifica la huella al arrancar, y si algún día no coincide **corta** en
vez de detectar mal en silencio.

In [ ]:
ruta = os.path.join(OBJ, "checkpoints", "head_best_solo.pt")
print(f"{ruta}  ({os.path.getsize(ruta)/1e6:.1f} MB)")
if EN_COLAB:
    shutil.copy(ruta, DEST)     # queda en Drive
    from google.colab import files
    files.download(ruta)
else:
    print("en Kaggle: bajalo desde el panel de Output")